In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
torch.set_grad_enabled(False)
torch.set_default_dtype(dtype)
torch.set_printoptions(precision=30, sci_mode=False)

print(f"Device: {device}")

In [ ]:
from typing import Any, Optional


def generate_inputs(
    n_concepts: int,
    l: float,
    u: float,
    device: Optional[Any] = None,
) -> torch.Tensor:
    return torch.stack(
        [
            torch.full((n_concepts,), l, device=device),
            torch.full((n_concepts,), u, device=device),
        ],
        dim=0,
    )


def generate_input_samples(
    n_samples: int,
    n_concepts: int,
    l: float,
    u: float,
    device: Optional[Any] = None,
) -> torch.Tensor:
    return torch.rand(n_samples, n_concepts, device=device) * (u - l) + l


def upwards_rounding(x: torch.Tensor) -> torch.Tensor:
    return torch.nextafter(x, torch.tensor(float("inf")))


def downwards_rounding(x: torch.Tensor) -> torch.Tensor:
    return torch.nextafter(x, torch.tensor(float("-inf")))


def outwards_rounding(x: torch.Tensor) -> torch.Tensor:
    assert x.shape[0] == 2, "The first dimmension of the tensor has not size 2"

    x[0] = torch.nextafter(x[0], torch.tensor(float("-inf")))
    x[1] = torch.nextafter(x[1], torch.tensor(float("inf")))

    return x


def is_inside_state(
    state_1: torch.Tensor, state_2: torch.Tensor, tolerance: float = 0.0
) -> bool:
    """
    Checks if concrete state_1 is a subset (is contained in) of state_2

    Args:
        state_1 (torch.Tensor): Tensor of shape (2, n_concepts)
        state_2 (torch.Tensor): Tensor of shape (2, n_concepts)

    Returns:
        bool: True if state_1 is is completely contained in state_2
    """

    assert (
        state_1.shape[1] == state_2.shape[1]
    ), f"State series has different number of concepts ({state_1.shape[2]} != {state_2.shape[2]})"

    is_contained = (state_2[0, :] - tolerance <= state_1[0, :]) & (
        state_1[1, :] <= state_2[1, :] + tolerance
    )

    return bool(is_contained.all().item())


def state_series_from_samples(
    sampled_state_series: torch.Tensor, device: Optional[Any] = None
) -> torch.Tensor:
    """
    Creates a state series from a set of sampled state series

    Args:
        sampled_state_series (torch.Tensor): Tensor of shape (n_samples, n_iters, n_concepts)

    Returns:
        torch.Tensor: The resulting state series. A tensor of shape (2, n_iters, n_concepts)
    """
    min_values = torch.min(sampled_state_series, dim=0).values
    max_values = torch.max(sampled_state_series, dim=0).values

    return torch.stack([min_values, max_values], dim=1)


def is_inside_state_series(
    state_series_1: torch.Tensor, state_series_2: torch.Tensor, tolerance: float = 0.0
) -> bool:
    """
    Checks if concrete state_series_1 is a subset (is contained in) of state_series_2

    Args:
        state_series_1 (torch.Tensor): Tensor of shape (n_iters, 2, n_concepts)
        state_series_2 (torch.Tensor): Tensor of shape (n_iters, 2, n_concepts)

    Returns:
        bool: True if state_series_1 is is completely contained in state_series_2
    """

    assert (
        state_series_1.shape[0] == state_series_2.shape[0]
    ), f"State series has different lengths ({state_series_1.shape[0]} != {state_series_2.shape[0]})"

    assert (
        state_series_1.shape[2] == state_series_2.shape[2]
    ), f"State series has different number of concepts ({state_series_1.shape[2]} != {state_series_2.shape[2]})"

    is_contained = (state_series_2[:, 0, :] - tolerance <= state_series_1[:, 0, :]) & (
        state_series_1[:, 1, :] <= state_series_2[:, 1, :] + tolerance
    )

    return bool(is_contained.all().item())


def states_intersection(state1: torch.Tensor, state2: torch.Tensor) -> torch.Tensor:
    return outwards_rounding(
        torch.stack(
            [
                torch.maximum(state1[0], state2[0]),
                torch.minimum(state1[1], state2[1]),
            ],
            dim=0,
        )
    )


def covering(state_series: torch.Tensor, iteration=-1) -> float:
    """
    Calculates the covering on a given iteration

    Args:
        state_series (torch.Tensor): A tensor of shape (n_iters, 2, n_concepts)
        iteration (int, optional): The iteration on which the covering is calculated. Defaults to -1.

    Returns:
        float: The calculated value
    """
    n_concepts = state_series[iteration].shape[1]

    return (
        torch.sum(
            (state_series[iteration, 1, :] - state_series[iteration, 0, :])
            / (state_series[0, 1, :] - state_series[0, 0, :])
        ).item()
        / n_concepts
    )


def average_gap(
    state_series: torch.Tensor, simulations: torch.Tensor, iteration: int = -1
):
    # all_states: [iterations, 2 (min/max), neurons]
    # history: [stimuli, iterations, neurons]
    min_neuron_pred = state_series[iteration][0]  # predicted minimum per neuron
    max_neuron_pred = state_series[iteration][1]  # predicted maximum per neuron

    min_neuron_val, _ = simulations[:, iteration, :].min(
        dim=0
    )  # min observed activation per neuron
    max_neuron_val, _ = simulations[:, iteration, :].max(
        dim=0
    )  # max observed activation per neuron

    avg_gap = (min_neuron_val - min_neuron_pred + max_neuron_pred - max_neuron_val) / 2
    return torch.mean(avg_gap)

In [ ]:
from typing import Any, Callable, Optional, Tuple

from experimental.tensor_functions import (
    ActivationFunction,
    line_from_points_vectorized,
    lower_quasi_nonlinear_sigmoid_step,
    lower_quasi_nonlinear_tanh_step,
    quasi_nonlinear_sigmoid,
    quasi_nonlinear_sigmoid_derivative,
    quasi_nonlinear_tanh,
    quasi_nonlinear_tanh_derivative,
    tangent_to_function_vectorized,
    upper_quasi_nonlinear_sigmoid_step,
    upper_quasi_nonlinear_tanh_step,
)


def get_function_data(
    activation_function: ActivationFunction, phi: float
) -> Tuple[Callable, Callable, Callable, Callable]:
    func, derivative, lower_step, upper_step = {
        "sigmoid": (
            quasi_nonlinear_sigmoid(phi),
            quasi_nonlinear_sigmoid_derivative(phi),
            lower_quasi_nonlinear_sigmoid_step(phi),
            upper_quasi_nonlinear_sigmoid_step(phi),
        ),
        "tanh": (
            quasi_nonlinear_tanh(phi),
            quasi_nonlinear_tanh_derivative(phi),
            lower_quasi_nonlinear_tanh_step(phi),
            upper_quasi_nonlinear_tanh_step(phi),
        ),
    }[activation_function]

    return func, derivative, lower_step, upper_step


def lower_relaxation(
    inputs: torch.Tensor,
    activation_function: ActivationFunction,
    phi: float,
    n_iter: int = 10,
):
    n_concepts = inputs.shape[1]
    inputs_shape = (2, n_concepts)

    assert (
        inputs_shape == inputs.shape
    ), f"inputs must have shape {inputs_shape}, but has shape {inputs.shape}"

    func, derivative, lower_step, _ = get_function_data(activation_function, phi)

    func_1 = func(inputs[1])
    derivative_0 = derivative(inputs[0])

    m1, b1 = line_from_points_vectorized(
        torch.stack([inputs[0], func(inputs[0])], dim=0),
        torch.stack([inputs[1], func(inputs[1])], dim=0),
    )
    m2, b2 = tangent_to_function_vectorized(
        (inputs[0] + inputs[1]) / 2, func, derivative
    )

    mask1 = m1 <= derivative_0
    m = torch.where(mask1, m1, torch.zeros_like(m1))
    b = torch.where(mask1, b1, torch.zeros_like(b1))

    mask2 = (~mask1) & (m2 * inputs[1] + b2 <= func_1)
    m = torch.where(mask2, m2, m1)
    b = torch.where(mask2, b2, b1)

    mask3 = (~mask1) & (~mask2)

    if mask3.any():
        z_iter = inputs[0][mask3].clone()
        for _ in range(n_iter):
            z_iter = lower_step(z_iter, inputs[1][mask3])

        m[mask3], b[mask3] = tangent_to_function_vectorized(z_iter, func, derivative)

    return m, b


def upper_relaxation(
    inputs: torch.Tensor,
    activation_function: ActivationFunction,
    phi: float,
    n_iter: int = 10,
):
    n_concepts = inputs.shape[1]
    input_shape = (2, n_concepts)
    assert (
        inputs.shape == input_shape
    ), f"inputs must have shape {input_shape}, got {inputs.shape}"

    func, derivative, _, upper_step = get_function_data(activation_function, phi)

    func_0 = func(inputs[0])
    derivative_1 = derivative(inputs[1])

    m1, b1 = line_from_points_vectorized(
        torch.stack([inputs[0], func_0], dim=0),
        torch.stack([inputs[1], func(inputs[1])], dim=0),
    )
    m2, b2 = tangent_to_function_vectorized(
        (inputs[0] + inputs[1]) / 2, func, derivative
    )

    mask1 = m1 <= derivative_1
    m = torch.where(mask1, m1, torch.zeros_like(m1))
    b = torch.where(mask1, b1, torch.zeros_like(b1))

    mask2 = (~mask1) & (m2 * inputs[0] + b2 >= func_0)
    m = torch.where(mask2, m2, m1)
    b = torch.where(mask2, b2, b1)

    mask3 = (~mask1) & (~mask2)

    if mask3.any():
        z_iter = inputs[1][mask3].clone()
        for _ in range(n_iter):
            z_iter = upper_step(z_iter, inputs[0][mask3])

        m[mask3], b[mask3] = tangent_to_function_vectorized(z_iter, func, derivative)

    return m, b


def induced_state_space(
    n_concepts: int,
    device: Optional[Any] = None,
) -> torch.Tensor:
    """
    Builds the induced state space
    """

    indices = torch.arange(n_concepts, device=device)
    base = torch.zeros((n_concepts, 2 * n_concepts + 1), device=device)
    base[indices, indices + 1] = 1

    return torch.stack([base, base])


def concretize_state_space(
    states: torch.Tensor,
    inputs: torch.Tensor,
    original_inputs: Optional[torch.Tensor] = None,
    device: Optional[Any] = None,
) -> torch.Tensor:
    """
    Concretiza el estado simbólico usando los inputs, de forma vectorizada.

    Args:
        states: tensor (n_iters, 2, n_concepts, n_concepts + 1)
        inputs: tensor (2, n_concepts)

    Returns:
        result: tensor (2, n_concepts)
    """
    n_concepts = states.shape[2]
    state_shape = (2, n_concepts, 2 * n_concepts + 1)
    input_shape = (2, n_concepts)

    assert (
        states.shape[1:] == state_shape
    ), f"states must have shape {state_shape}, got {states.shape[1:]}"
    assert (
        inputs.shape == input_shape
    ), f"inputs must have shape {input_shape}, got {inputs.shape}"

    if original_inputs is None:
        original_inputs = inputs

    inputs = outwards_rounding(torch.cat([inputs, original_inputs], dim=1))

    input_coeffs = torch.zeros_like(states, device=device)
    input_coeffs[:, :, :, 0] = 1.0

    positive_elements_mask = states[:, :, :, 1:] > 0

    inputs_i = inputs[None, :, None, :]
    inputs_j = inputs.flip(dims=[0])[None, :, None, :]

    input_coeffs[:, :, :, 1:] = torch.where(positive_elements_mask, inputs_i, inputs_j)

    input_coeffs[:, 0, :, :] = downwards_rounding(input_coeffs[:, 0, :, :])
    input_coeffs[:, 1, :, :] = upwards_rounding(input_coeffs[:, 1, :, :])

    # print(input_coeffs.shape)

    result = torch.sum(states * input_coeffs, dim=3)
    result[:, 0, :] = downwards_rounding(result[:, 0, :])
    result[:, 1, :] = upwards_rounding(result[:, 1, :])

    return result


def propagate_state_space(state: torch.Tensor, W: torch.Tensor):
    n_concepts = W.shape[0]
    w_shape = (n_concepts, n_concepts)
    state_shape_prefix = (2, n_concepts)

    assert (
        state.shape[:2] == state_shape_prefix
    ), f"state shape must have shape {state_shape_prefix} in the first two dimensions, but has shape {state.shape}"

    assert W.shape == w_shape, f"W must have shape {w_shape}, but has shape {W.shape}"

    W_pos = torch.clamp(W, min=0)
    W_neg = torch.clamp(W, max=0)

    result = outwards_rounding(
        torch.matmul(W_pos, state) + torch.matmul(W_neg, state.flip(dims=[0]))
    )

    return result


def apply_relaxed_activation(
    state: torch.Tensor,
    inputs: torch.Tensor,
    activation_function: ActivationFunction,
    phi=1.0,
    device: Optional[Any] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    n_concepts = state.shape[1]
    state_shape = (2, n_concepts, 2 * n_concepts + 1)
    intervals_shape = (2, n_concepts)

    assert (
        state.shape == state_shape
    ), f"state must have shape {state_shape}, but has shape {state.shape}"

    assert (
        inputs.shape == intervals_shape
    ), f"inputs must have shape {intervals_shape} but has shape {inputs.shape}"

    lower_m, lower_b = lower_relaxation(inputs, activation_function, phi)
    upper_m, upper_b = upper_relaxation(inputs, activation_function, phi)

    lower_m = downwards_rounding(lower_m)
    lower_b = downwards_rounding(lower_b)
    upper_m = upwards_rounding(upper_m)
    upper_b = upwards_rounding(upper_b)

    m_coeffs = torch.stack([lower_m, upper_m], dim=0).unsqueeze(-1)
    b_coeffs = torch.stack([lower_b, upper_b], dim=0)

    result = outwards_rounding(m_coeffs * state)
    result[:, :, 0] += b_coeffs
    result = outwards_rounding(result)

    concept_indexes = torch.arange(n_concepts, device=device)
    result[:, concept_indexes, concept_indexes + n_concepts + 1] += 1.0 - phi
    result = outwards_rounding(result)

    return result, lower_m, lower_b, upper_m, upper_b


def symbolic_iteration(
    state: torch.Tensor,
    W: torch.Tensor,
    inputs: torch.Tensor,
    phi: float,
    activation_function: ActivationFunction,
    original_inputs: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    new_state = propagate_state_space(state, W)
    concrete_state = concretize_state_space(
        torch.unsqueeze(new_state, dim=0),
        inputs,
        original_inputs=original_inputs,
        device=device,
    )
    concrete_state = concrete_state[0]

    new_state = apply_relaxed_activation(
        new_state, concrete_state, activation_function, phi=phi
    )[0]
    return outwards_rounding(new_state)


def arithmetic_iteration(
    W_pos: torch.Tensor,
    W_neg: torch.Tensor,
    state: torch.Tensor,
    activation_function: ActivationFunction,
    phi: float,
    inputs: Optional[torch.Tensor] = None,
):
    if inputs is None:
        inputs = state

    transfer_function = (
        torch.sigmoid if activation_function == "sigmoid" else torch.tanh
    )

    new_state = torch.stack(
        [
            (torch.matmul(W_pos, state[i]) + torch.matmul(W_neg, state[1 - i]))
            for i in range(2)
        ],
        dim=0,
    )
    new_state = phi * transfer_function(new_state) + (1 - phi) * inputs

    return outwards_rounding(new_state)


def containment_guided_state_space(
    W: torch.Tensor,
    inputs: torch.Tensor,
    activation_function: ActivationFunction,
    phi: float,
    minimal_symbolic_warming_iters: int = 5,
    arithmetic_rectifications: int = 5,
    max_iters: int = 20,
    baseline_interval_series: Optional[torch.Tensor] = None,
    device: Optional[Any] = None,
):
    n_concepts = W.shape[0]
    w_shape = (n_concepts, n_concepts)
    inputs_shape = (2, n_concepts)

    W_pos = torch.clamp(W, min=0)
    W_neg = torch.clamp(W, max=0)

    assert W.shape == w_shape, f"W must have shape {w_shape}, but has shape {W.shape}"

    assert (
        inputs.shape == inputs_shape
    ), f"inputs must have shape {inputs_shape} but has shape {inputs.shape}"

    if baseline_interval_series is not None:
        assert baseline_interval_series.shape[1:] == (2, n_concepts)

    all_states = torch.empty(
        (max_iters + 1, 2, n_concepts, 2 * n_concepts + 1), device=device
    )
    concrete_states = torch.empty((max_iters + 1, 2, n_concepts), device=device)
    naive_states = torch.empty((max_iters + 1, 2, n_concepts), device=device)

    all_states[0] = induced_state_space(n_concepts, device=device)
    concrete_states[0] = inputs
    naive_states[0] = inputs

    index = 1
    current_inputs_index = 0
    pending_symbolic_iters = minimal_symbolic_warming_iters

    while index <= max_iters:
        state = symbolic_iteration(
            all_states[index - 1],
            W,
            concrete_states[current_inputs_index],
            phi,
            activation_function=activation_function,
            original_inputs=inputs,
        )
        concrete_state = concretize_state_space(
            torch.unsqueeze(state, dim=0),
            concrete_states[current_inputs_index],
            device=device,
            original_inputs=inputs,
        ).squeeze(dim=0)
        concrete_state = outwards_rounding(concrete_state)

        concrete_states[index] = arithmetic_iteration(
            W_pos, W_neg, concrete_states[index - 1], activation_function, phi, inputs
        )

        if pending_symbolic_iters > 0 or is_inside_state(
            concrete_state, concrete_states[index]
        ):
            all_states[index] = state
            concrete_states[index] = states_intersection(
                concrete_states[index], concrete_state
            )

            if baseline_interval_series is not None:
                concrete_states[index] = states_intersection(
                    concrete_states[index], baseline_interval_series[index]
                )

            index += 1
            pending_symbolic_iters -= 1

        else:
            for _ in range(min(arithmetic_rectifications, max_iters - index + 1)):
                all_states[index] = induced_state_space(n_concepts, device=device)
                concrete_states[index] = arithmetic_iteration(
                    W_pos,
                    W_neg,
                    concrete_states[index - 1],
                    activation_function,
                    phi,
                    inputs,
                )
                if baseline_interval_series is not None:
                    concrete_states[index] = states_intersection(
                        concrete_states[index], baseline_interval_series[index]
                    )
                current_inputs_index = index
                index += 1

            pending_symbolic_iters = minimal_symbolic_warming_iters

    return concrete_states


def interval_state_space(
    W: torch.Tensor,
    inputs: torch.Tensor,
    activation_function: ActivationFunction,
    phi: float,
    max_iters: int = 20,
    device: Optional[Any] = None,
) -> torch.Tensor:
    W_pos = torch.clamp(W, min=0)
    W_neg = torch.clamp(W, max=0)

    transfer_function = (
        torch.sigmoid if activation_function == "sigmoid" else torch.tanh
    )

    all_states = torch.empty((max_iters + 1, *inputs.shape), device=device)
    all_states[0] = inputs.clone()

    for t in range(1, max_iters + 1):
        all_states[t] = torch.stack(
            [
                (
                    torch.matmul(W_pos, all_states[t - 1][i])
                    + torch.matmul(W_neg, all_states[t - 1][1 - i])
                )
                for i in range(2)
            ],
            dim=0,
        )
        all_states[t] = phi * transfer_function(all_states[t]) + (1 - phi) * inputs

    return all_states


def rescaled_reasoning(
    A: torch.Tensor,
    W: torch.Tensor,
    T: int,
    phi: float,
    transfer_function: Callable[[torch.Tensor], torch.Tensor] = torch.sigmoid,
    device: Optional[torch.device] = None,
) -> torch.Tensor:
    """
    Implements the Sigmoid-Rescaled reasoning rule for Fuzzy Cognitive Maps in PyTorch.

    Args:
        A (torch.Tensor): Initial state matrix of shape (n_samples, n_concepts)
        W (torch.Tensor): Weight matrix of shape (n_concepts, n_concepts)
        T (int): Number of reasoning steps
        phi (float): Mixing parameter between transfer function output and initial state
        transfer_function (callable, optional): Activation function (torch version)
        device (torch.device, optional): Device for computation
        dtype (torch.dtype, optional): Type of tensor

    Returns:
        torch.Tensor: Tensor of shape (n_samples, T+1, n_concepts) with all states
    """

    n_samples, n_concepts = A.shape
    H = torch.empty((n_samples, T + 1, n_concepts), device=device)
    H[:, 0] = A

    for t in range(1, T + 1):
        H[:, t] = phi * transfer_function(H[:, t - 1] @ W) + (1 - phi) * H[:, 0]

    return H

In [ ]:
def rectified_intersection_state_space(
    W: torch.Tensor,
    inputs: torch.Tensor,
    activation_function: ActivationFunction,
    phi: float,
    max_iters: int = 20,
    minimal_warming_iters: int = 5,
):
    n_concepts = W.shape[0]

    default_state = torch.stack(
        [torch.full((n_concepts,), -torch.inf), torch.full((n_concepts,), torch.inf)]
    )

    rectified_intersection = rectified_intersection = default_state.unsqueeze(0).repeat(
        max_iters + 1, 1, 1
    )

    for arithmetic_rectifications in range(1, 11):
        rectified_states = containment_guided_state_space(
            W,
            inputs,
            activation_function,
            phi=phi,
            max_iters=max_iters,
            device=device,
            minimal_symbolic_warming_iters=minimal_warming_iters,
            arithmetic_rectifications=arithmetic_rectifications,
            baseline_interval_series=rectified_intersection,
        )

        for i in range(max_iters + 1):
            rectified_intersection[i] = states_intersection(
                rectified_intersection[i], rectified_states[i]
            )
    return rectified_intersection

In [ ]:
import numpy as np
from core.utils import generate_random_matrix
from luis.plot import compare_state_evolutions_and_simulations


MAX_ITERS = 50

n_concepts = 10
density = 0.8
activation: ActivationFunction = "sigmoid"  # "tanh"
# activation: ActivationFunction = "tanh"

phi = 0.7
n_samples = 100
l_0 = 0
u_0 = 1
MINIMAL_WARMING = 5

seed = int(np.random.randint(0, int(1e5)))
# seed = 74287

print(f"SEED = {seed}")

np.random.seed(seed)
torch.manual_seed(seed)

W = generate_random_matrix(n_concepts, density, device=device, dtype=dtype)

Wt = W.T
inputs = generate_inputs(n_concepts, l_0, u_0, device=device).to(dtype=dtype)

naive_states = interval_state_space(
    Wt,
    inputs,
    max_iters=MAX_ITERS,
    phi=phi,
    activation_function=activation,
    device=device,
)

A = generate_input_samples(n_samples, n_concepts, l_0, u_0, device=device)

simulated_states = rescaled_reasoning(
    A,
    W,
    T=MAX_ITERS,
    phi=phi,
    transfer_function=torch.sigmoid if activation == "sigmoid" else torch.tanh,
    device=device,
)

sampled_states = state_series_from_samples(simulated_states)

rectified_states1 = containment_guided_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    max_iters=MAX_ITERS,
    device=device,
    minimal_symbolic_warming_iters=MINIMAL_WARMING,
    naive_rectifications=1,
    baseline_interval_series=naive_states,
)

print(f"Naive Covering: {covering(naive_states)}")
print(f"Rectified Covering: {covering(rectified_states1)}")

compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    rectified_states1.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Symbolic",
)


In [ ]:
# Primera version:

default_state = torch.stack(
    [torch.full((n_concepts,), -torch.inf), torch.full((n_concepts,), torch.inf)]
)

rectified_intersection = rectified_intersection = default_state.unsqueeze(0).repeat(
    MAX_ITERS + 1, 1, 1
)

for arithmetic_rectifications in range(1, 11):
    rectified_states = containment_guided_state_space(
        Wt,
        inputs,
        activation,
        phi=phi,
        max_iters=MAX_ITERS,
        device=device,
        minimal_symbolic_warming_iters=MINIMAL_WARMING,
        arithmetic_rectifications=arithmetic_rectifications,
        baseline_interval_series=naive_states,
    )

    for i in range(MAX_ITERS + 1):
        rectified_intersection[i] = states_intersection(
            rectified_intersection[i], rectified_states[i]
        )

print(
    f"Samples lie inside rectified states 1: {is_inside_state_series(sampled_states, rectified_states1, tolerance=1e-6)}"
)


print(f"Naive Covering: {covering(naive_states)}")
print(f"Rectified Covering: {covering(rectified_states1)}")
print(f"Multi-Rectified Covering 1: {covering(rectified_intersection)}")

# Segunda version

rectified_intersection = rectified_intersection_state_space(
    Wt, inputs, activation, phi, MAX_ITERS, minimal_warming_iters=MINIMAL_WARMING
)


print(f"Multi-Rectified Covering 2: {covering(rectified_intersection)}")

print(
    f"Samples lie inside rectified states 2: {is_inside_state_series(sampled_states, rectified_intersection, tolerance=1e-6)}"
)

compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    rectified_intersection.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "rectified_intersection",
)

In [ ]:
gap = average_gap(rectified_intersection, simulated_states)

print(f"Gap: {gap}")